# HHL algorithm


## Imports


In [9]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator

## 1. Step 1: State Preparation


In [4]:
def encode_b(b_vec: np.ndarray) -> tuple[QuantumCircuit, float]:
	"""
	Encode a real 2-element vector into a single qubit using Ry rotation.
	Returns the circuit and the angle used.
	"""
	b_vec = b_vec / np.linalg.norm(b_vec)
	theta = 2 * np.arctan2(b_vec[1], b_vec[0])

	qc = QuantumCircuit(1)
	qc.ry(theta, 0)
	return qc, theta


def verify_state_preparation():
	sim = AerSimulator(method='statevector')
	test_cases = [
		np.array([1.0, 0.0]),
		np.array([0.0, 1.0]),
		np.array([1.0, 1.0]),
		np.array([3.0, 1.0]),
	]

	print("=== Step 1: State Preparation ===")
	for b_vec in test_cases:
		qc, _ = encode_b(b_vec)
		qc.save_statevector() # type:ignore
		sv = sim.run(qc).result().get_statevector()

		expected = b_vec / np.linalg.norm(b_vec)
		match = np.allclose(np.array(sv), expected, atol=1e-6)
		print(f"  b={b_vec} -> {'PASS' if match else 'FAIL'}")
  
verify_state_preparation()

=== Step 1: State Preparation ===
  b=[1. 0.] -> PASS
  b=[0. 1.] -> PASS
  b=[1. 1.] -> PASS
  b=[3. 1.] -> PASS


## 2. Step 2: Quantum Phase Estimation


In [44]:
def build_qpe(b_angle: float, t: float = np.pi / 2, n_clock: int = 2) -> QuantumCircuit:
	"""
	Build the full QPE circuit for A = 1.5*I + 0.5*X.

	Qubit layout:
		b[0]       : encodes |b>
		clock[0,1] : clock register (stores eigenvalue estimate)

	Args:
		b_angle  : Ry angle to prepare the b register (from encode_b)
		t        : base evolution time; pi/2 maps eigenvalues 1,2 -> clock states 1,2
		n_clock  : number of clock qubits
	"""
	b_reg     = QuantumRegister(1,       'b')
	clock_reg = QuantumRegister(n_clock, 'clock')
	qc        = QuantumCircuit(b_reg, clock_reg)

	# 1. Prepare |b>
	qc.ry(b_angle, b_reg[0])
	qc.barrier()

	# 2. Hadamard on all clock qubits -> superposition over all clock states
	for i in range(n_clock):
		qc.h(clock_reg[i])
	qc.barrier()

	# 3. Controlled-U^(2^k) for each clock qubit k
	#    U = e^{iAt}, A = 1.5*I + 0.5*X
	#    Decomposition:
	#      - P(1.5 * angle) on clock[k]   <- handles the e^{i*1.5t}*I factor
	#      - CRx(-angle)    on b[0]        <- handles the e^{i*0.5t*X} factor
	for k in range(n_clock):
		angle = t * (2 ** k)
		qc.p(1.5 * angle, clock_reg[k])          # global phase kickback
		qc.crx(-angle, clock_reg[k], b_reg[0])   # Rx rotation on b
	qc.barrier()

	# 4. Inverse QFT on clock register
	iqft = QFTGate(n_clock).inverse()
	qc.append(iqft, clock_reg)

	return qc


def verify_qpe():
	"""
	Test QPE by feeding in each eigenvector of A and checking the clock register.

	Eigenvectors of A = 1.5*I + 0.5*X:
		|u1> = (|0> + |1>) / sqrt(2),  eigenvalue 1  -> clock state |01> = 1
		|u2> = (|0> - |1>) / sqrt(2),  eigenvalue 2  -> clock state |10> = 2

	Ry angles:
		|u1>: theta =  pi/2   (cos pi/4 |0> + sin pi/4 |1>)
		|u2>: theta = -pi/2   (cos pi/4 |0> - sin pi/4 |1>)
	"""
	sim = AerSimulator(method='statevector')

	test_cases = [
		("lambda=1  |u1>=(|0>+|1>)/sqrt2",  np.pi / 2,  2),
		("lambda=2  |u2>=(|0>-|1>)/sqrt2", -np.pi / 2,  1),
	]

	print("\n=== Step 2: QPE Verification ===")
	for label, angle, expected_clock in test_cases:
		qc = build_qpe(b_angle=angle)
		qc.save_statevector() # type:ignore
		sv = np.array(sim.run(qc.decompose()).result().get_statevector())
		probs = np.abs(sv) ** 2

		# Qiskit statevector index (little-endian): bit0=b, bit1=clock[0], bit2=clock[1]
		# clock integer = clock[0] + 2*clock[1]
		clock_probs = {}
		for idx, prob in enumerate(probs):
			c0 = (idx >> 1) & 1
			c1 = (idx >> 2) & 1
			# reverse bit order: clock[1] is MSB, clock[0] is LSB
			clock_int = c0 + 2 * c1
			clock_probs[clock_int] = clock_probs.get(clock_int, 0) + prob

		print(f"\n  {label}")
		print(f"  Expected clock state: {expected_clock}")
		for state, prob in sorted(clock_probs.items()):
			if prob > 1e-6:
				print(f"    clock={state} ({state:02b}) : prob={prob:.4f}")

		assert clock_probs.get(expected_clock, 0) > 0.99, \
			f"FAIL: expected clock={expected_clock}, got {clock_probs}"
		print(f"  PASS")

	print("\nQPE verification passed.")

verify_qpe()


=== Step 2: QPE Verification ===

  lambda=1  |u1>=(|0>+|1>)/sqrt2
  Expected clock state: 2
    clock=2 (10) : prob=1.0000
  PASS

  lambda=2  |u2>=(|0>-|1>)/sqrt2
  Expected clock state: 1
    clock=1 (01) : prob=1.0000
  PASS

QPE verification passed.


## 3. Step 3: Inverse eigenvalue


In [48]:
def build_eigenvalue_inversion(qc, clock, ancilla):
    """
    Uses X gates to convert target clock states into all-|1> for mcry control.
    """
    C = 1.0

    # --- lambda=1: clock state 2 (c0=0, c1=1) ---
    theta_1 = 2 * np.arcsin(C / 2.0)  # pi
    # Flip c0 so state (c0=0,c1=1) becomes (c0=1,c1=1) -> standard ccRy
    qc.x(clock[0])
    qc.mcry(theta_1, [clock[0], clock[1]], ancilla)
    qc.x(clock[0])  # unflip

    # --- lambda=2: clock state 1 (c0=1, c1=0) ---
    theta_2 = 2 * np.arcsin(C / 1.0)  # pi/3
    # Flip c1 so state (c0=1,c1=0) becomes (c0=1,c1=1) -> standard ccRy
    qc.x(clock[1])
    qc.mcry(theta_2, [clock[0], clock[1]], ancilla)
    qc.x(clock[1])  # unflip

    return qc


def verify_eigenvalue_inversion():
    """
    Test by preparing clock in a known state and checking ancilla rotation.
    """
    simulator = AerSimulator(method='statevector')

    test_cases = [
        {
            "clock_state": 2,   # lambda=1, expect ancilla ~ |1> (theta=pi)
            "expected_prob_1": 1.0,
            "label": "lambda=1 (clock=2)"
        },
        {
            "clock_state": 1,   # lambda=2, expect ancilla ~ sin^2(pi/6) = 0.25
            "expected_prob_1": 0.25,
            "label": "lambda=2 (clock=1)"
        },
    ]

    for tc in test_cases:
        b_reg     = QuantumRegister(1, 'b')
        clock_reg = QuantumRegister(2, 'clock')
        anc_reg   = QuantumRegister(1, 'ancilla')
        qc = QuantumCircuit(b_reg, clock_reg, anc_reg)

        # Manually set clock to desired state
        # clock_state=2 -> binary 10 -> clock[1]=1, clock[0]=0
        # clock_state=1 -> binary 01 -> clock[1]=0, clock[0]=1
        if tc["clock_state"] & 1:   # bit 0
            qc.x(clock_reg[1])
        if tc["clock_state"] & 2:   # bit 1
            qc.x(clock_reg[0])

        build_eigenvalue_inversion(qc, clock_reg, anc_reg[0])

        # Save statevector
        qc.save_statevector() # type:ignore
        job    = simulator.run(qc)
        sv     = job.result().get_statevector()
        probs  = sv.probabilities_dict()

        # Ancilla is the last qubit (index 3), sum probs where last bit = 1
        prob_anc_1 = sum(v for k, v in probs.items() if k[0] == '1')

        print(f"\n{tc['label']}")
        print(f"  Expected P(ancilla=1) = {tc['expected_prob_1']:.4f}")
        print(f"  Got      P(ancilla=1) = {prob_anc_1:.4f}")
        assert abs(prob_anc_1 - tc["expected_prob_1"]) < 1e-6, \
            f"FAILED: expected {tc['expected_prob_1']}, got {prob_anc_1}"
        print("  PASSED")
        
verify_eigenvalue_inversion()


lambda=1 (clock=2)
  Expected P(ancilla=1) = 1.0000
  Got      P(ancilla=1) = 1.0000
  PASSED

lambda=2 (clock=1)
  Expected P(ancilla=1) = 0.2500
  Got      P(ancilla=1) = 0.2500
  PASSED


## 4. Step 4: Inverse QPE

In [43]:
def build_inverse_qpe(t: float = np.pi / 2, n_clock: int = 2) -> QuantumCircuit:
    """
    Inverse QPE: uncomputes the clock register back to |00>.
    
    Reverses the QPE operations:
    1. Apply forward QFT (undoes inverse QFT)
    2. Apply controlled-U^dagger in reverse order
    3. Apply Hadamards
    """
    b_reg     = QuantumRegister(1,       'b')
    clock_reg = QuantumRegister(n_clock, 'clock')
    qc        = QuantumCircuit(b_reg, clock_reg)

    # Step 1: Forward QFT (undoes the inverse QFT from QPE)
    qft_fwd = QFTGate(n_clock)
    qc.append(qft_fwd, clock_reg)

    # Step 2: Controlled-U^dagger in reverse order
    # U = exp(iAt) where A = 1.5I + 0.5X
    # U^dagger = exp(-iAt)
    # 
    # Forward QPE did:
    #   for k in [0, 1, ...]:
    #       P(+1.5*angle) then CRx(-angle)
    #
    # Inverse should reverse both the order AND the operations:
    #   for k in [..., 1, 0]:
    #       CRx(+angle) then P(-1.5*angle)
    
    for k in reversed(range(n_clock)):
        angle = t * (2 ** k)
        # Apply inverse operations in reverse order
        qc.crx(angle, clock_reg[k], b_reg[0])  # Inverse of CRx(-angle)
        qc.p(-1.5 * angle, clock_reg[k])        # Inverse of P(+1.5*angle)

    # Step 3: Hadamards (self-inverse)
    qc.h(clock_reg)

    return qc


def verify_inverse_qpe(t: float = np.pi / 2, n_clock: int = 2):
    """
    Verifies that QPE followed by inverse QPE returns clock to |00>
    for both eigenstates of A.
    """
    # Eigenstates of A = 1.5I + 0.5X are |+> and |->
    # |+> has eigenvalue 2, |-> has eigenvalue 1
    test_cases = [
        ("eigenstate |+> (lambda=2)", np.pi / 2),   # Ry(pi/2)|0> = |+>
        ("eigenstate |-> (lambda=1)", -np.pi / 2),  # Ry(-pi/2)|0> = |->
    ]

    simulator = AerSimulator(method='statevector')

    for label, b_angle in test_cases:
        # Build QPE then inverse QPE back-to-back
        b_reg     = QuantumRegister(1,       'b')
        clock_reg = QuantumRegister(n_clock, 'clock')
        qc        = QuantumCircuit(b_reg, clock_reg)

        # Forward QPE (without state prep — we add it manually)
        qpe = build_qpe(b_angle, t, n_clock)
        qc.compose(qpe, inplace=True)
        qc.barrier()

        # Inverse QPE
        iqpe = build_inverse_qpe(t, n_clock)
        qc.compose(iqpe, inplace=True)

        qc.save_statevector() # type:ignore
        result = simulator.run(transpile(qc, simulator)).result()
        sv = result.get_statevector()

        # Check clock register is back to |00>
        # Qubit order: b[0], clock[0], clock[1]
        # State index encodes: clock[1] clock[0] b[0] (little-endian)
        probs = {}
        for idx, amp in enumerate(sv):
            prob = abs(amp) ** 2
            if prob > 1e-6:
                bits = format(idx, f'0{1 + n_clock}b')
                # bits[0] = clock[1], bits[1] = clock[0], bits[2] = b[0]
                clock_bits = bits[:n_clock]  # leftmost n_clock bits
                probs[clock_bits] = probs.get(clock_bits, 0) + prob

        # All probability should be in clock state '00'
        clock_00_prob = probs.get('00', 0.0)
        passed = abs(clock_00_prob - 1.0) < 1e-4

        print(f"[{'PASS' if passed else 'FAIL'}] {label}")
        print(f"  Clock state probabilities: {probs}")
        print(f"  P(clock=00) = {clock_00_prob:.6f}")

verify_inverse_qpe()


[FAIL] eigenstate |+> (lambda=2)
  Clock state probabilities: {'10': np.float64(0.49999999999999983), '11': np.float64(0.4999999999999999)}
  P(clock=00) = 0.000000
[FAIL] eigenstate |-> (lambda=1)
  Clock state probabilities: {'10': np.float64(0.49999999999999994), '11': np.float64(0.49999999999999956)}
  P(clock=00) = 0.000000


/tmp/ipykernel_11522/1329277617.py:79: DeprecationWarning: The return type of saved statevectors has been changed from a `numpy.ndarray` to a `qiskit.quantum_info.Statevector` as of qiskit-aer 0.10. Accessing numpy array attributes is deprecated and will result in an error in a future release. To continue using saved result objects as arrays you can explicitly cast them using  `np.asarray(object)`.
  for idx, amp in enumerate(sv):


## 5. Step 5: Build HHL

In [57]:
def build_hhl(
    b_vec: np.ndarray,
    t: float = np.pi / 2,
    n_clock: int = 2,
    C: float = 1.0
) -> QuantumCircuit:
    """
    Full HHL circuit for 2x2 system Ax = b.

    Qubit layout:
        b[0]       : encodes the vector b
        clock[0,1] : clock register for QPE
        ancilla[0] : ancilla for eigenvalue inversion
    """
    b_reg       = QuantumRegister(1,       'b')
    clock_reg   = QuantumRegister(n_clock, 'clock')
    ancilla_reg = QuantumRegister(1,       'ancilla')
    qc          = QuantumCircuit(b_reg, clock_reg, ancilla_reg)

    # --- Step 1: State Preparation ---
    state_prep, _ = encode_b(b_vec)
    qc.compose(state_prep, qubits=[b_reg[0]], inplace=True)
    qc.barrier(label='State Prep')

    # --- Step 2: QPE ---
    qc.h(clock_reg)
    for k in range(n_clock):
        angle = t * (2 ** k)
        qc.p(1.5 * angle, clock_reg[k])
        qc.crx(-angle, clock_reg[k], b_reg[0])
    qft_inv = QFTGate(n_clock).inverse()
    qc.append(qft_inv, clock_reg)
    qc.barrier(label='QPE')

    # --- Step 3: Eigenvalue Inversion ---
    build_eigenvalue_inversion(qc, clock_reg, ancilla_reg[0])
    qc.barrier(label='Eigen Inv')

    # --- Step 4: Inverse QPE ---
    iqpe = build_inverse_qpe(t=t, n_clock=n_clock)
    qc.compose(iqpe, qubits=[*b_reg, *clock_reg], inplace=True)
    qc.barrier(label='iQPE')

    return qc

def verify_hhl(b_vec: np.ndarray, A: np.ndarray, t: float = np.pi / 2, n_clock: int = 2, C: float = 1.0):
    """
    Verifies HHL output against classical solution.
    Post-selects on ancilla=|1>, then checks b register matches A^{-1}b.
    """
    simulator = AerSimulator(method='statevector')

    qc = build_hhl(b_vec, t, n_clock, C)
    qc.save_statevector() # type:ignore

    result = simulator.run(transpile(qc, simulator)).result()
    sv = np.array(result.get_statevector())

    # Total qubits: b(1) + clock(n_clock) + ancilla(1)
    n_total = 1 + n_clock + 1

    # Post-select: keep only amplitudes where ancilla=|1>
    # Ancilla is the last qubit added => index 0 in the bit string (leftmost)
    post_selected = {}
    for idx, amp in enumerate(sv):
        bits = format(idx, f'0{n_total}b')
        # bits[0] = ancilla, bits[1..n_clock] = clock, bits[-1] = b
        ancilla_bit = bits[0]
        clock_bits  = bits[1:1 + n_clock]
        b_bit       = bits[-1]
        if ancilla_bit == '1' and clock_bits == '0' * n_clock:
            post_selected[b_bit] = amp

    # Reconstruct statevector of b register from post-selected amplitudes
    amp_0 = post_selected.get('0', 0.0)
    amp_1 = post_selected.get('1', 0.0)
    hhl_vec = np.array([amp_0, amp_1])

    # Normalize
    norm = np.linalg.norm(hhl_vec)
    hhl_vec_normalized = hhl_vec / norm

    # Classical solution
    x_classical = np.linalg.solve(A, b_vec)
    x_classical_normalized = x_classical / np.linalg.norm(x_classical)

    # Compare (up to global phase)
    overlap = abs(np.dot(hhl_vec_normalized, x_classical_normalized))
    passed  = abs(overlap - 1.0) < 1e-3

    print(f"[{'PASS' if passed else 'FAIL'}] HHL verification")
    print(f"""  A matrix         :\n{A}""")
    print(f"  b vector         : {b_vec}")
    print(f"  Classical x      : {x_classical_normalized}")
    print(f"  HHL x (normalized): {np.real(hhl_vec_normalized)}")
    print(f"  Overlap          : {overlap:.6f}")
    print(f"  Success prob     : {norm**2:.6f}")

# Test with A = 1.5I + 0.5X, b = [1, 0]
# A = 1.5 * np.eye(2) + 0.5 * np.array([[0,1],[1,0]])
A = np.array([[1.5, 0.5],[0.5, 1.5]])
b = np.array([0.0, 1.0])
verify_hhl(b, A)


[PASS] HHL verification
  A matrix         :
[[1.5 0.5]
 [0.5 1.5]]
  b vector         : [0. 1.]
  Classical x      : [-0.31622777  0.9486833 ]
  HHL x (normalized): [-0.31622777  0.9486833 ]
  Overlap          : 1.000000
  Success prob     : 0.625000


## 6. Diagnosis and Visual

In [27]:
def diagnose_hhl_steps(b_vec: np.ndarray, t: float = np.pi / 2, n_clock: int = 2):
    from qiskit_aer import AerSimulator

    simulator = AerSimulator(method='statevector')

    def get_sv(qc):
        qc_copy = qc.copy()
        qc_copy.save_statevector()           
        result = simulator.run(qc_copy.decompose(reps=5)).result()
        return result.get_statevector()

    def print_sv(sv, label):
        print(f"\n--- {label} ---")
        n = int(np.log2(len(sv)))
        for i, amp in enumerate(sv):
            if abs(amp) > 1e-6:
                bits = format(i, f'0{n}b')[::-1]  # little-endian -> MSB first display
                print(f"  |{bits}⟩ (b,c0,c1,anc): {amp:.4f}  |amp|={abs(amp):.4f}")

    b_reg       = QuantumRegister(1,       'b')
    clock_reg   = QuantumRegister(n_clock, 'clock')
    ancilla_reg = QuantumRegister(1,       'ancilla')

    # Step 1: encode_b
    qc = QuantumCircuit(b_reg, clock_reg, ancilla_reg)
    state_prep, b_angle = encode_b(b_vec)
    qc.compose(state_prep, qubits=[b_reg[0]], inplace=True)
    print_sv(get_sv(qc), "After encode_b")

    # Step 2: QPE
    qc.h(clock_reg)
    for k in range(n_clock):
        angle = t * (2 ** k)
        qc.p(1.5 * angle, clock_reg[k])
        qc.crx(-angle, clock_reg[k], b_reg[0])
    qft_inv = QFTGate(n_clock).inverse()
    qc.append(qft_inv, clock_reg)
    print_sv(get_sv(qc), "After QPE")

    # Step 3: Eigenvalue inversion
    build_eigenvalue_inversion(qc, clock_reg, ancilla_reg[0])
    print_sv(get_sv(qc), "After Eigenvalue Inversion")

    # Step 4: Inverse QPE
    iqpe = build_inverse_qpe(t=t, n_clock=n_clock)
    qc.compose(iqpe, qubits=[*b_reg, *clock_reg], inplace=True)
    print_sv(get_sv(qc), "After Inverse QPE")

    print("\n--- Expected after inverse QPE ---")
    print("  Clock register should be |00⟩ for all terms")
    print("  Ancilla |1⟩ terms give the solution after post-selection")


def visualize_circuit(b_vector: np.ndarray, t: float = np.pi / 2, n_clock: int = 2):
    """
    Generate and display the full HHL circuit.
    """
    qc = build_hhl(b_vector, t, n_clock)
    
    # Draw circuit
    print("Full HHL Circuit:")
    print("=" * 80)
    print(qc.draw(output='text', fold=-1))
    print("=" * 80)
    print(f"Circuit depth: {qc.depth()}")
    print(f"Gate count: {qc.size()}")
    
    return qc


# Test with b = [1, 0] (eigenvector with eigenvalue 1)
print("\n" + "="*80)
print("TEST CASE: b = [1, 0] (eigenvector, λ=1)")
print("="*80)
b_test = np.array([1.0, 1.0])
diagnose_hhl_steps(b_test)


print("\n" + "="*80)
print("CIRCUIT VISUALIZATION")
print("="*80)
visualize_circuit(b_test)



TEST CASE: b = [1, 0] (eigenvector, λ=1)

--- After encode_b ---
  |0000⟩ (b,c0,c1,anc): 0.7071+0.0000j  |amp|=0.7071
  |1000⟩ (b,c0,c1,anc): 0.7071+0.0000j  |amp|=0.7071

--- After QPE ---
  |0010⟩ (b,c0,c1,anc): 0.7071-0.0000j  |amp|=0.7071
  |1010⟩ (b,c0,c1,anc): 0.7071-0.0000j  |amp|=0.7071

--- After Eigenvalue Inversion ---
  |0011⟩ (b,c0,c1,anc): 0.7071-0.0000j  |amp|=0.7071
  |1011⟩ (b,c0,c1,anc): 0.7071-0.0000j  |amp|=0.7071

--- After Inverse QPE ---
  |0001⟩ (b,c0,c1,anc): 0.7071-0.0000j  |amp|=0.7071
  |1001⟩ (b,c0,c1,anc): 0.7071-0.0000j  |amp|=0.7071

--- Expected after inverse QPE ---
  Clock register should be |00⟩ for all terms
  Ancilla |1⟩ terms give the solution after post-selection

CIRCUIT VISUALIZATION
Full HHL Circuit:
         ┌─────────┐ State Prep                 ┌──────────┐┌────────┐            QPE                                                                         Eigen Inv         ┌───────┐┌─────────┐                   iQPE 
      b: ┤ Ry(π/2) ├─────

/tmp/ipykernel_11522/3469925258.py:14: DeprecationWarning: The return type of saved statevectors has been changed from a `numpy.ndarray` to a `qiskit.quantum_info.Statevector` as of qiskit-aer 0.10. Accessing numpy array attributes is deprecated and will result in an error in a future release. To continue using saved result objects as arrays you can explicitly cast them using  `np.asarray(object)`.
  n = int(np.log2(len(sv)))
/tmp/ipykernel_11522/3469925258.py:15: DeprecationWarning: The return type of saved statevectors has been changed from a `numpy.ndarray` to a `qiskit.quantum_info.Statevector` as of qiskit-aer 0.10. Accessing numpy array attributes is deprecated and will result in an error in a future release. To continue using saved result objects as arrays you can explicitly cast them using  `np.asarray(object)`.
  for i, amp in enumerate(sv):
